In [ ]:
import numpy as np
import pandas as pd
import sympy as sp
from pathlib import Path
from main import run_ga
import main as ga_main

In [ ]:
NGUYEN_SPECS = [
    {
        "id": 1,
        "func": lambda x: x**3 + x**2 + x,
        "expr": "x**3 + x**2 + x",
        "variables": [{"name": "x", "sampler": (-5.0, 5.0)}],
        "allowed_unary": [],
    },
    {
        "id": 2,
        "func": lambda x: x**4 + x**3 + x**2 + x,
        "expr": "x**4 + x**3 + x**2 + x",
        "variables": [{"name": "x", "sampler": (-5.0, 5.0)}],
        "allowed_unary": [],
    },
    {
        "id": 3,
        "func": lambda x: x**5 + x**4 + x**3 + x**2 + x,
        "expr": "x**5 + x**4 + x**3 + x**2 + x",
        "variables": [{"name": "x", "sampler": (-5.0, 5.0)}],
        "allowed_unary": [],
    },
    {
        "id": 4,
        "func": lambda x: x**6 + x**5 + x**4 + x**3 + x**2 + x,
        "expr": "x**6 + x**5 + x**4 + x**3 + x**2 + x",
        "variables": [{"name": "x", "sampler": (-5.0, 5.0)}],
        "allowed_unary": [],
    },
    {
        "id": 5,
        "func": lambda x: np.sin(x**2) * np.cos(x) - 1,
        "expr": "np.sin(x**2)*np.cos(x) - 1",
        "variables": [{"name": "x", "sampler": (-2.0, 2.0)}],
        "allowed_unary": ["sin", "cos"],
    },
    {
        "id": 6,
        "func": lambda x: np.sin(x) + np.sin(x + x**2),
        "expr": "np.sin(x) + np.sin(x + x**2)",
        "variables": [{"name": "x", "sampler": (-2.0, 2.0)}],
        "allowed_unary": ["sin"],
    },
    {
        "id": 7,
        "func": lambda x: np.log(x + 1) + np.log(x**2 + 1),
        "expr": "np.log(x + 1) + np.log(x**2 + 1)",
        "variables": [{"name": "x", "sampler": (0.0, 50.0)}],
        "allowed_unary": ["log"],
    },
    {
        "id": 8,
        "func": lambda x: np.sqrt(np.abs(x)),
        "expr": "np.sqrt(np.abs(x))",
        "variables": [{"name": "x", "sampler": (-5.0, 5.0)}],
        "allowed_unary": ["abs", "sqrt"],
    },
    {
        "id": 9,
        "func": lambda x, y: np.sin(x) + np.sin(y**2),
        "expr": "np.sin(x) + np.sin(y**2)",
        "variables": [
            {"name": "x", "sampler": (-1.0, 1.0)},
            {"name": "y", "sampler": (-1.0, 1.0)},
        ],
        "allowed_unary": ["sin"],
    },
    {
        "id": 10,
        "func": lambda x, y: 2 * np.sin(x) * np.cos(y),
        "expr": "2*np.sin(x)*np.cos(y)",
        "variables": [
            {"name": "x", "sampler": (-1.0, 1.0)},
            {"name": "y", "sampler": (-1.0, 1.0)},
        ],
        "allowed_unary": ["sin", "cos"],
    },
    {
        "id": 11,
        "func": lambda x, y: x**y,
        "expr": "x**y",
        "variables": [
            {"name": "x", "sampler": (0.1, 3.0)},
            {"name": "y", "sampler": (0.1, 3.0)},
        ],
        "allowed_unary": [],
        "allow_pow": True,
    },
    {
        "id": 12,
        "func": lambda x, y: x**4 - x**3 + (y**2) / 2 - y,
        "expr": "x**4 - x**3 + (y**2)/2 - y",
        "variables": [
            {"name": "x", "sampler": (-2.5, 2.5)},
            {"name": "y", "sampler": (-2.5, 2.5)},
        ],
        "allowed_unary": [],
    },
]

In [ ]:
SEED = 1

ga_config = {
    "pop_size": 1000,
    "gens": 75,
    "max_init_depth": 5,
    "max_depth": 25,
    "crossover_rate": 0.85,
    "mutation_rate": 0.3,
    "tournament_k": 20,
    "elitism": 8,
    "use_learned_crossover": True,
    "selector_lr": 0.0003,
    "mlp_layers": 12,
    "mlp_hidden_dim": 128,
    "mlp_activation": "leaky_relu",
    "mlp_dropout": 0.1,
    "use_critic": True,
    "stopping_metric": "r2",
    "stopping_threshold": 1.0,
    "complexity_coef": 2e-4,
    "inclusion_coef": 1.0,
    "prob_const": 0.3,
    "const_range": (-5, 5),
    "point_mutation_prob": 0.1,
    "simplify_new_genomes": False,
    "simplify_timeout": 0.2,
    "selector_update_freq": 3,
    "temperature": 0.75,
    "entropy_weight": 0.075,
    "opt_constants": True,
    "const_opt_freq": 3,
    "const_opt_method": "L-BFGS-B",
    "const_opt_maxiter": 300,
    "verbose": True,
    "use_pareto": True,
    "print_freq": 1,
    "seed": SEED,
}

# Basic unary functions expected by run_ga: (numpy_func, sympy_func) tuples
default_unary_funcs = {
    "sin": (np.sin, sp.sin),
    "cos": (np.cos, sp.cos),
    "exp": (np.exp, sp.exp),
    "log": (np.log, sp.log),
    "sqrt": (np.sqrt, sp.sqrt),
    "abs": (np.abs, sp.Abs),
}

# Per-benchmark unary maps (provide sympy conversion alongside numpy callables)
nguyen_unary_funcs = dict(default_unary_funcs)

# Result collectors & sample sizes
nguyen_results = []

NGUYEN_SAMPLE_SIZE = 512

In [ ]:
# Nguyen equation 1
nguyen_score_1, nguyen_result_1 = run_equation(
    NGUYEN_SPECS[0],
)
nguyen_result_1

In [ ]:
# Nguyen equation 2
nguyen_score_2, nguyen_result_2 = run_equation(
    NGUYEN_SPECS[1],
)
nguyen_result_2

In [ ]:
# Nguyen equation 3
nguyen_score_3, nguyen_result_3 = run_equation(
    NGUYEN_SPECS[2],
)
nguyen_result_3

In [ ]:
# Nguyen equation 4
nguyen_score_4, nguyen_result_4 = run_equation(
    NGUYEN_SPECS[3],
)
nguyen_result_4

In [ ]:
# Nguyen equation 5
nguyen_score_5, nguyen_result_5 = run_equation(
    NGUYEN_SPECS[4],
)
nguyen_result_5

In [ ]:
# Nguyen equation 6
nguyen_score_6, nguyen_result_6 = run_equation(
    NGUYEN_SPECS[5],
)
nguyen_result_6

In [ ]:
# Nguyen equation 7
nguyen_score_7, nguyen_result_7 = run_equation(
    NGUYEN_SPECS[6],
)
nguyen_result_7

In [ ]:
# Nguyen equation 8
nguyen_score_8, nguyen_result_8 = run_equation(
    NGUYEN_SPECS[7],
)
nguyen_result_8

In [ ]:
# Nguyen equation 9
nguyen_score_9, nguyen_result_9 = run_equation(
    NGUYEN_SPECS[8],
)
nguyen_result_9

In [ ]:
# Nguyen equation 10
nguyen_score_10, nguyen_result_10 = run_equation(
    NGUYEN_SPECS[9],
)
nguyen_result_10

In [ ]:
# Nguyen equation 11
nguyen_score_11, nguyen_result_11 = run_equation(
    NGUYEN_SPECS[10],
)
nguyen_result_11

In [ ]:
# Nguyen equation 12
nguyen_score_12, nguyen_result_12 = run_equation(
    NGUYEN_SPECS[11],
)
nguyen_result_12